# NB6 — Final validation-AUC training V5 — FROZEN

**Status:** selected final V1 development run. Ranking-loss and scheduler experiments are not part of this frozen V5.

Frozen run:
- seed `42`
- BCEWithLogitsLoss only
- standard sample-level shuffled batches
- FP32 training
- FP32 validation
- initial/fixed LR `3e-4`
- `max_epochs = 60`
- `early_stopping_patience = 10`
- `early_stopping_min_epochs = 30`
- checkpoint selection by validation ROC-AUC only
- test split was **not loaded**

Frozen best checkpoint:
- best epoch: **52**
- validation ROC-AUC: **0.6905082489625538**
- FITB 2-way: **0.7626970227670753**
- mean logit margin: **1.0499372052358245**
- median logit margin: **0.7358774170279503**
- validation BCE loss: **0.6972924366515071**
- validation samples / paired families: **2284 / 1142**

The selected run was produced from Git commit `7cbbb19fd89352b7ef54038e57b4d8208b7ee1f6`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.colab import drive
drive.mount("/content/drive")

ARTIFACT_ROOT = Path("/content/drive/MyDrive/ML_Final")
os.environ["FASHION_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["FASHION_EMBEDDING_CACHE"] = str(ARTIFACT_ROOT / "fashionclip_item_embeddings.pt")
os.environ["FASHION_EMBEDDING_MANIFEST"] = str(ARTIFACT_ROOT / "embedding_manifest_v1.json")
os.environ["FASHION_CORE7_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "core7_drop_v2")
os.environ["FASHION_SCORER_READY_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "scorer_ready_v2")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)

HEAD = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()
print("Git HEAD:", HEAD)

In [ ]:
# Regression checks. unittest discovery avoids collisions with third-party
# packages named `tests` in notebook environments.
test_run = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-p",
        "test_scorer*.py",
        "-v",
    ],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode != 0:
    raise RuntimeError(
        f"Scorer regression tests failed with exit code {test_run.returncode}"
    )
print("SCORER REGRESSION TESTS: PASS")

In [ ]:
import yaml
import torch

from src.data.runtime_paths import load_runtime_paths
from src.scorer.checkpoint import build_runtime_provenance, load_checkpoint
from src.scorer.model import TypeAwarePairwiseScorer
from src.scorer.train import (
    build_train_valid_loaders,
    evaluate_epoch,
    fit_scorer,
    seed_everything,
    validate_s3_config,
)

CONFIG_PATH = REPO_ROOT / "configs" / "scorer_type_aware_pairwise_v1_val_auc.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

validate_s3_config(config)
training = config["training"]

assert training["mixed_precision"] is False
assert training["learning_rate"] == 0.0003
assert training["max_epochs"] == 60
assert training["early_stopping_patience"] == 10
assert training["early_stopping_min_epochs"] == 30
assert training["lr_scheduler"] == "none"
assert training["seed"] == 42
assert config["selection"]["primary_metric"] == "roc_auc"

paths = load_runtime_paths(repo_root=REPO_ROOT)
provenance = build_runtime_provenance(paths, REPO_ROOT)
assert provenance["git_tree_clean"] is True

loaders = build_train_valid_loaders(paths, config, num_workers=0)
train_dataset = loaders["datasets"]["train"]
valid_dataset = loaders["datasets"]["valid"]
train_loader = loaders["train_loader"]
valid_loader = loaders["valid_loader"]

assert len(train_dataset) == 30918
assert len(valid_dataset) == 2284
assert len(train_dataset.pair_families) == 15459
assert len(valid_dataset.pair_families) == 1142

print("CONFIG / DATA / PROVENANCE: PASS")

In [ ]:
SEED = int(training["seed"])
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Use a GPU runtime; training remains FP32."

model = TypeAwarePairwiseScorer.from_config(config).to(device)
with torch.no_grad():
    category_weights = model.category_embedding.weight.detach()
    category_norm_mean = float(category_weights[1:].norm(dim=1).mean())
    pad_norm = float(category_weights[0].norm())

assert model.category_embedding_init_policy == "post_mlp_scale_preserving"
assert 0.7 < category_norm_mean < 1.3
assert pad_norm == 0.0

RUN_DIR = (
    paths.artifact_root
    / "scorer_runs"
    / "type_aware_pairwise_v1"
    / "final_val_auc_v5_seed42"
)
print("Run dir:", RUN_DIR)
print("Category norm mean:", category_norm_mean)
print("MODEL INIT: PASS")

In [ ]:
# Re-training is intentionally guarded: never overwrite the frozen V5 run.
if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    raise RuntimeError(
        f"Frozen V5 run already exists at {RUN_DIR}. "
        "Do not overwrite it. Use the evaluation cell below."
    )

result = fit_scorer(
    model,
    train_loader,
    valid_loader,
    config=config,
    checkpoint_dir=RUN_DIR,
    provenance=provenance,
    device=device,
)
print("Best epoch:", result["best_epoch"])
print("Best validation ROC-AUC:", result["best_valid_roc_auc"])

In [ ]:
# Canonical evaluation of frozen best.pt in FP32.
BEST_PATH = RUN_DIR / "best.pt"
assert BEST_PATH.is_file(), f"Missing frozen checkpoint: {BEST_PATH}"

best_model = TypeAwarePairwiseScorer.from_config(config).to(device)
best_payload = load_checkpoint(
    BEST_PATH,
    model=best_model,
    map_location=device,
    current_provenance=provenance,
)
best_model.eval()

criterion = torch.nn.BCEWithLogitsLoss()
best_valid = evaluate_epoch(
    best_model,
    valid_loader,
    criterion=criterion,
    device=device,
)

print("FINAL BEST VALIDATION")
for key, value in best_valid.items():
    print(f"{key:26s}: {value}")

assert best_payload["epoch"] == 52
assert abs(float(best_valid["roc_auc"]) - 0.6905082489625538) < 1e-12
assert abs(float(best_valid["fitb_2way"]) - 0.7626970227670753) < 1e-12
assert best_valid["sample_count"] == 2284
assert best_valid["paired_family_count"] == 1142

print("FROZEN V5 CHECKPOINT: VERIFIED")
print("TEST SPLIT WAS NOT LOADED.")